In [ ]:
from PIL import Image
import numpy as np
import os
from constants import ImageMetadata, Header, PIPE_PATH, NEG_MODE, SLICE_MODE
import json
from tkinter import filedialog
from tkinter import Tk

In [ ]:
def load_metadata_from_dialog():
    """Open a file picker and return image metadata plus color flag.

    Returns:
        tuple[ImageMetadata, bool] | None: Image metadata with raw bytes and
        whether the selected image is colored (PPM), or None if canceled.
    """
    # Create a hidden Tk root only to display the file chooser dialog.
    root = Tk()
    root.withdraw()

    # Accept grayscale PGM or color PPM input images.
    image_path = filedialog.askopenfilename(
        title="Select a file",
        filetypes=[("Gray image files", "*.pgm"), ("Color image files", "*.ppm")]
    )
    

    if not image_path:
        print("No file selected")
        return None

    # Determine if the selected image is RGB based on extension.
    colored = ".ppm" in image_path

    with Image.open(image_path) as img:
        # Convert image to bytes and preserve metadata required by the worker.
        arr = np.array(img)
        data = arr.tobytes()
        return ImageMetadata(img.height, img.width, int(arr.max()), data), colored

In [ ]:
def load_header_from_input():
    """Collect processing mode parameters from terminal input.

    Returns:
        Header: Header configured for negative mode or slice-threshold mode.
    """
    while True:
        try:
            # Ask which operation mode will be executed by the worker.
            mode = int(input(f"Choose mode ({NEG_MODE}=NEG, {SLICE_MODE}=SLICE): ").strip())
            if mode not in (NEG_MODE, SLICE_MODE):
                print("Invalid mode. Try again.")
                continue

            if mode == NEG_MODE:
                # NEG mode ignores slice limits.
                return Header(mode=mode, slice_range=(0, 0))

            # SLICE mode needs the inclusive/exclusive row interval.
            start = int(input("Slice start (row index): ").strip())
            end = int(input("Slice end (row index, exclusive): ").strip())
            if start < 0 or end <= start:
                print("Invalid slice range. Use start >= 0 and end > start.")
                continue

            return Header(mode=mode, slice_range=(start, end))
        except ValueError:
            print("Please enter integer values.")

In [ ]:
def ask_metadata_and_start_pipe(metadata, header):
    """Send metadata header and raw image bytes to the worker via named pipe.

    Args:
        metadata (ImageMetadata | None): Selected image metadata and bytes.
        header (Header | None): Processing parameters to apply on worker side.
    """
    if metadata is None or header is None:
        return

    # Create named pipe once; worker opens the same path for reading.
    if not os.path.exists(PIPE_PATH):
        os.mkfifo(PIPE_PATH)
        print(f"Created named pipe at {PIPE_PATH}")

    print("Sender process started. Opening pipe...")

    with open(PIPE_PATH, "wb") as pipe:
        # Send JSON control payload first, then raw image bytes.
        payload = metadata.to_dict()
        payload["header"] = header.to_dict()
        message = json.dumps(payload).encode("utf-8")
        pipe.write(message + b"\n")
        pipe.write(metadata.data)
        pipe.flush()

    print("Sender process finished")

In [ ]:
def main():
    """Run the sender flow: choose image, read mode, and send payload."""
    # Load image bytes + basic metadata from a file dialog.
    metadata, colored = load_metadata_from_dialog()
    if metadata:
        # Load operation parameters and attach color mode before sending.
        header = load_header_from_input()
        header.colored = colored
        ask_metadata_and_start_pipe(metadata, header)

In [ ]:
if __name__ == "__main__":
    main()